## 工具

#### 工具调用

In [23]:
from typing import Literal

from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取城市的天气

    Args:
        city: 城市名称
    """
    return f"{city}天气是晴朗的，温度是25摄氏度"

@tool(parse_docstring=True)
def get_news() -> str:
    """ 获取新闻 """
    return "最近有新的大模型发布"

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

tools = [get_weather, get_news]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from rich.pretty import pprint


class State(MessagesState):
    output: str

def llm_node(state: State):
    ai_msg = model_with_tools.invoke(state["messages"])
    return {
        "messages": [ai_msg],
    }

def tool_node(state: State):
    last_msg = state["messages"][-1]
    if not isinstance(last_msg, AIMessage) or not last_msg.tool_calls:
        return {}

    tool_msgs = []
    for tool_call in last_msg.tool_calls:
        tool = tools_by_name[tool_call["name"]]
        tool_result = tool.invoke(tool_call)
        tool_msgs.append(tool_result)

    return {
        "messages": tool_msgs,
    }

def route(state: State) -> Literal["tool_node", END]:
    last_msg = state["messages"][-1]
    if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
        return "tool_node"
    return END


builder = StateGraph(State)
builder.add_node(llm_node)
builder.add_node(tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", route)
builder.add_edge("tool_node", "llm_node")
graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage("北京天气如何？最近有啥新闻")]})
pprint(result)

##### 使用 ToolNode 实现工具调用

移除 tool_node，使用 ToolNode 实现工具调用

In [ ]:
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import StateGraph, START, MessagesState
from rich.pretty import pprint


class State(MessagesState):
    output: str

def llm_node(state: State):
    ai_msg = model_with_tools.invoke(state["messages"])
    return {
        "messages": [ai_msg],
    }

def route(state: State) -> Literal["tool_node", END]:
    last_msg = state["messages"][-1]
    if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
        return "tool_node"
    return END


builder = StateGraph(State)
builder.add_node(llm_node)
# 添加 ToolNode 节点
builder.add_node("tool_node", ToolNode(tools))
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", route)
builder.add_edge("tool_node", "llm_node")
graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage("北京天气如何？最近有啥新闻")]})
pprint(result)